# 教程: 检查和理解模型状态

## 目的
水文模型不仅仅是输入和输出之间的黑箱。模型内部维护着一系列**状态变量（State Variables）**，这些变量描述了系统在任一时刻的“记忆”或物理状况，例如土壤的含水量、水库的蓄水量等。理解这些内部状态如何随时间演变，对于深入理解模型行为、调试问题和建立对模型的信任至关重要。

此笔记本将：
1.  选择一个包含明确内部状态的简单水文模型。
2.  在模拟过程中，不仅记录模型的最终输出，还记录其关键内部状态变量的值。
3.  通过绘图，将输入、输出和内部状态的演变过程并列展示。
4.  分析这些图表，以揭示输入、状态和输出之间的因果关系。

## 1. 环境设置和模型准备

我们选择由`SimpleRunoffModule`和`SimpleRouting`组成的模型。这个组合有两个非常直观的状态变量：
- `SimpleRunoffModule.S`: 代表土壤蓄水层的当前蓄水量。
- `SimpleRouting.Q_s`: 代表慢速流（基流）部分的当前蓄水量。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from hydro_model.model import HydrologicalModel
from hydro_model.runoff import SimpleRunoffModule
from hydro_model.routing import SimpleRouting

# --- 定义模型和输入 ---
params = {'S_max': 200, 'c_loss': 0.05, 'k_q': 0.7, 'k_s': 0.2}
rainfall = np.array([0, 0, 10, 20, 50, 30, 15, 10, 5, 0, 0, 0, 0, 25, 40, 10, 0, 0])
pet = np.full_like(rainfall, 2.0) # 假设每天蒸散发2mm

## 2. 运行模拟并记录状态

我们实例化模型，并在一个循环中运行它。关键在于，在每个时间步调用`model.run()`之后，我们立即访问并记录下`runoff_module.S`和`routing_module.Q_s`的值。

In [ ]:
runoff_module = SimpleRunoffModule(**params)
routing_module = SimpleRouting(**params)
model = HydrologicalModel(runoff_module, routing_module)

num_steps = len(rainfall)
state_history = {
    'outflow': [],
    'soil_storage': [],
    'slow_flow_storage': []
}

print("运行模拟并记录状态...")
for i in range(num_steps):
    # 运行一步
    outflow = model.run(rainfall[i], pet[i])
    
    # 记录状态
    state_history['outflow'].append(outflow)
    state_history['soil_storage'].append(runoff_module.S)
    state_history['slow_flow_storage'].append(routing_module.Q_s)

print("模拟完成。")

## 3. 可视化输入、状态和输出

现在我们将所有记录下来的时间序列数据绘制在一张包含多个子图的图表上，以便进行对比分析。

In [ ]:
timesteps = np.arange(num_steps)
fig, axes = plt.subplots(4, 1, figsize=(12, 12), sharex=True)
fig.suptitle('Inspecting Model States Over Time', fontsize=16)

# 1. 降雨输入
axes[0].bar(timesteps, rainfall, color='c', label='Rainfall')
axes[0].set_ylabel('Rainfall (mm)')
axes[0].legend()
axes[0].grid(True, linestyle='--')

# 2. 土壤蓄水状态 (S)
axes[1].plot(timesteps, state_history['soil_storage'], 'g-', label='Soil Storage (S)')
axes[1].axhline(params['S_max'], color='r', linestyle=':', label=f"S_max = {params['S_max']}")
axes[1].set_ylabel('Storage (mm)')
axes[1].legend()
axes[1].grid(True, linestyle='--')

# 3. 慢速流蓄水状态 (Q_s)
axes[2].plot(timesteps, state_history['slow_flow_storage'], 'm-', label='Slow Flow Storage (Q_s)')
axes[2].set_ylabel('Storage (mm)')
axes[2].legend()
axes[2].grid(True, linestyle='--')

# 4. 总出流
axes[3].plot(timesteps, state_history['outflow'], 'b-', label='Total Outflow')
axes[3].set_xlabel('Time Step (days)')
axes[3].set_ylabel('Flow (mm)')
axes[3].legend()
axes[3].grid(True, linestyle='--')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### 结果解读

通过并列观察这四张图，我们可以清晰地看到模型内部的因果链条：
- **降雨与土壤蓄水**: 每次下雨时，土壤蓄水（`S`，绿线）都会上升。当降雨停止后，`S`会因为蒸发（`pet`）和损失（`c_loss`）而缓慢下降。
- **土壤蓄水与产流**: 在第一次降雨事件中（t=2到t=8），土壤蓄水从0开始增加，因此产流（反映在`outflow`的快速上升部分和`Q_s`的增加）相对温和。在第二次降雨事件中（t=13开始），由于土壤已经具有一定的底墒（`S`>0），因此产流响应更快、更强。
- **慢速流蓄水与基流**: 慢速流蓄水（`Q_s`，紫线）是对产流中慢速部分的累积，它以一个更慢的速率消退，构成了总出流（蓝线）中的“基流”部分，使得总出流过程线更为平滑。

通过这种方式检查内部状态，我们可以确认模型的行为是否符合我们的物理直觉和水文学知识，这对于模型开发和调试至关重要。